# 9. `boot (command <N>)` (action, start)

In [1]:
from common import *

shape, log = show(9)

log shape 7: action | start | boot (command <N>)
13,075 logs (3.2%) from 256 nodes, 2003-12-26 18:24:10 to 2006-04-24 21:18:09
log 29619 at 2006-04-20 16:47:15: 459874 node-98 action start 1145551635 1 boot  (command 4220)


Node 98 was rebooted. Why? Was it automatic or manual?

What the node logged in the day before

In [2]:
node_day = timeline(log.node, log.ts - day, log.ts)
display(node_day)
went_down = node_day[node_day.template == "not responding"].iloc[0]
print(f"not responding at {went_down.first_ts}, booted at {log.ts}. Looks like manual operation after the node stopped responding.")

,first_ts,last_ts,count,component,state,template
0,2006-04-20 13:42:02,2006-04-20 13:42:02,1,node,temperature,ambient=<N>
1,2006-04-20 14:09:36,2006-04-20 14:09:36,1,node,status,not responding
2,2006-04-20 14:09:40,2006-04-20 14:09:40,1,node,status,configured out
3,2006-04-20 16:47:15,2006-04-20 16:47:15,1,action,start,boot (command <N>)


not responding at 2006-04-20 14:09:36, booted at 2006-04-20 16:47:15. Looks like manual operation after the node stopped responding.


Some extra information not required for the verdict: the command number (4220) is a source of logs

In [3]:
as_frame(logs.search(log.ts - hour, log.ts + hour, node=4220, limit=None))

,id,ts,node,component,state,message,log_shape_id
0,33877,2006-04-20 16:46:26,4220,boot_cmd,new,Targeting domains:node-D3 and nodes:node-[96-103] child of command 4208,20
1,31791,2006-04-20 16:53:19,4220,boot_cmd,success,Command has completed successfully,14


Nodes that stopped responding in the 4 hours before, and nodes booted within the hour

In [4]:
down = logs.stats(log.ts - 4 * hour, log.ts, log_shape_id=NOT_RESPONDING)
booted = logs.stats(log.ts - hour, log.ts + hour, log_shape_id=shape.log_shape_id)
print("not responding:", down)
print("boot commands: ", booted)
assert down.n_nodes == 256 and booted.n_nodes == 256

not responding: LogStats(n_logs=256, n_nodes=256, first_seen='2006-04-20 14:04:50', last_seen='2006-04-20 14:15:00')
boot commands:  LogStats(n_logs=258, n_nodes=256, first_seen='2006-04-20 16:47:14', last_seen='2006-04-20 17:41:06')


# Verdict: no action, this line is the action.
All 256 nodes had stopped responding, and operators boot them all; the command for this node reports success.
- In hindsight it tells what the fix for the outage was and how long the node was down.